## PACOTES ##

In [ ]:
import time
import itertools
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    precision_recall_curve,
    auc,
    log_loss
)

## CÓDIGO ##

In [ ]:
def truncar_6(x):
    return np.trunc(float(x) * 1_000_000) / 1_000_000

df = pd.read_csv("creditcard.csv")

if "status_fraude" in df.columns:
    target_name = "status_fraude"

elif "Class" in df.columns:
    df = df.rename(columns={"Class": "status_fraude"})
    target_name = "status_fraude"

else:
    raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

features = [
    col for col in df.columns
    if col != target_name
    and pd.api.types.is_numeric_dtype(df[col])
]

combinacoes_2x2 = list(itertools.combinations(features, 2))

print("Dataset carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")
print(f"Target utilizado: {target_name}")
print(f"Quantidade de features numéricas: {len(features)}")
print(f"Quantidade de combinações 2x2: {len(combinacoes_2x2)}")


def logpdf_gaussiana_multivariada(X, media, cov):

    X = np.asarray(X)
    media = np.asarray(media)
    cov = np.asarray(cov)

    if X.ndim == 1:
        X = X.reshape(-1, 1)

    if media.ndim == 0:
        media = np.array([media])

    media = media.reshape(-1)

    cov = np.atleast_2d(cov)

    n_features = X.shape[1]

    sinal, logdet = np.linalg.slogdet(cov)

    if sinal <= 0:
        return np.full(X.shape[0], -np.inf)

    diff = X - media

    solucao = np.linalg.solve(cov, diff.T).T

    termo_quadratico = np.sum(
        diff * solucao,
        axis=1
    )

    logpdf = -0.5 * (
        n_features * np.log(2 * np.pi)
        + logdet
        + termo_quadratico
    )

    return logpdf

def calcular_log_veross_com_rotulo(
    X_scaled,
    y_real,
    reg_covar=1e-6
):

    X_scaled = np.asarray(X_scaled)

    if X_scaled.ndim == 1:
        X_scaled = X_scaled.reshape(-1, 1)

    y_real = np.asarray(y_real).astype(int)

    n_amostras, n_features = X_scaled.shape

    log_veross_total = 0.0

    for classe in [0, 1]:

        X_classe = X_scaled[y_real == classe]

        n_classe = X_classe.shape[0]

        if n_classe <= 1:
            return np.nan

        peso_classe = n_classe / n_amostras

        media_classe = np.mean(
            X_classe,
            axis=0
        )

        cov_classe = np.cov(
            X_classe,
            rowvar=False
        )

        cov_classe = np.atleast_2d(cov_classe)

        cov_classe = cov_classe + reg_covar * np.eye(n_features)

        logpdf_classe = logpdf_gaussiana_multivariada(
            X=X_classe,
            media=media_classe,
            cov=cov_classe
        )

        log_veross_total += np.sum(
            np.log(peso_classe) + logpdf_classe
        )

    log_veross_media = log_veross_total / n_amostras

    return log_veross_media

def encontrar_melhor_ponto_corte_mcc(y_real, probabilidades):

    y_real = np.asarray(y_real).astype(int)
    probabilidades = np.asarray(probabilidades)

    ordem = np.argsort(-probabilidades)

    probs_ord = probabilidades[ordem]
    y_ord = y_real[ordem]

    total_positivos = np.sum(y_ord == 1)
    total_negativos = np.sum(y_ord == 0)

    tp_acum = np.cumsum(y_ord == 1)
    fp_acum = np.cumsum(y_ord == 0)

    fn_acum = total_positivos - tp_acum
    tn_acum = total_negativos - fp_acum

    numerador = (tp_acum * tn_acum) - (fp_acum * fn_acum)

    denominador = np.sqrt(
        (tp_acum + fp_acum) *
        (tp_acum + fn_acum) *
        (tn_acum + fp_acum) *
        (tn_acum + fn_acum)
    )

    mccs = np.divide(
        numerador,
        denominador,
        out=np.zeros_like(numerador, dtype=float),
        where=denominador != 0
    )

    indices_validos = np.r_[
        np.where(probs_ord[:-1] != probs_ord[1:])[0],
        len(probs_ord) - 1
    ]

    mccs_validos = mccs[indices_validos]

    melhor_idx_local = np.argmax(mccs_validos)
    melhor_idx = indices_validos[melhor_idx_local]

    melhor_ponto_corte = probs_ord[melhor_idx]
    melhor_mcc = mccs[melhor_idx]

    return melhor_ponto_corte, melhor_mcc

def analisar_combinacao_2x2(
    df,
    feature_1,
    feature_2,
    target_name="status_fraude",
    resumo_combinacoes=None,
    verbose=False
):

    if resumo_combinacoes is None:
        resumo_combinacoes = []

    inicio = time.perf_counter()

    if verbose:
        print(f"\nIniciando combinação: {feature_1} + {feature_2}")

    temp = df[[feature_1, feature_2, target_name]].dropna()

    if temp.empty:
        if verbose:
            print("  - Ignorada: dados vazios após dropna")
        return resumo_combinacoes

    X = temp[[feature_1, feature_2]]
    y_real = temp[target_name].astype(int)

    if y_real.nunique() < 2:
        if verbose:
            print("  - Ignorada: target possui apenas uma classe")
        return resumo_combinacoes

    if X[feature_1].nunique() < 2 or X[feature_2].nunique() < 2:
        if verbose:
            print("  - Ignorada: uma das features é constante")
        return resumo_combinacoes

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    try:
        gmm = GaussianMixture(
            n_components=2,
            covariance_type="full",
            random_state=42,
            n_init=3,
            reg_covar=1e-6
        )

        gmm.fit(X_scaled)

    except Exception as erro:
        if verbose:
            print(f"  - Erro no treinamento da GMM: {erro}")
        return resumo_combinacoes

    log_veross_gmm = gmm.score(X_scaled)

    log_veross_com_rotulo = calcular_log_veross_com_rotulo(
        X_scaled=X_scaled,
        y_real=y_real,
        reg_covar=1e-6
    )

    neg_log_veross_com_rotulo = -log_veross_com_rotulo
    neg_log_veross_gmm = -log_veross_gmm

    diferenca_neg_log_veross = (
        neg_log_veross_com_rotulo
        - neg_log_veross_gmm
    )

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(
        clusters,
        y_real
    )

    if 1 not in ct.columns:
        if verbose:
            print("  - Ignorada: classe 1 não encontrada no crosstab")
        return resumo_combinacoes

    cluster_fraude = ct[1].idxmax()

    probabilidades = gmm.predict_proba(X_scaled)[:, cluster_fraude]

    probabilidades = np.clip(
        probabilidades,
        1e-15,
        1 - 1e-15
    )

    precision_vals, recall_vals, _ = precision_recall_curve(
        y_real,
        probabilidades
    )

    auc_pr = auc(
        recall_vals,
        precision_vals
    )


    melhor_ponto_corte, mcc = encontrar_melhor_ponto_corte_mcc(
        y_real=y_real,
        probabilidades=probabilidades
    )

    ponto_corte_medio = 0.5


    ll = log_loss(
        y_real,
        probabilidades
    )

    fim = time.perf_counter()


    auc_pr_norm = np.clip(
        auc_pr,
        0,
        1
    )

    mcc_norm = (mcc + 1) / 2

    mcc_norm = np.clip(
        mcc_norm,
        0,
        1
    )

    log_loss_norm = 1 / (1 + ll)

    log_loss_norm = np.clip(
        log_loss_norm,
        0,
        1
    )

    score_final = np.mean([
        auc_pr_norm,
        mcc_norm,
        log_loss_norm
    ])


    nova_linha = {
        "Feature_1": feature_1,
        "Feature_2": feature_2,
        "Combinacao": f"{feature_1} + {feature_2}",

        "AUC_PR": truncar_6(float(auc_pr)),
        "MCC": truncar_6(float(mcc)),
        "Log_Loss": truncar_6(float(ll)),
        "Log_Loss_Norm": truncar_6(float(log_loss_norm)),

        "Neg_Log_Veross_Com_Rotulo": truncar_6(float(neg_log_veross_com_rotulo)),
        "Neg_Log_Veross_GMM": truncar_6(float(neg_log_veross_gmm)),
        "Diferenca_Neg_Log_Veross": truncar_6(float(diferenca_neg_log_veross)),

        "Score_Final": truncar_6(float(score_final)),
        "Melhor_Ponto_Corte": truncar_6(float(melhor_ponto_corte)),
        "Ponto_Corte_Medio": truncar_6(float(ponto_corte_medio)),
        "Tempo": truncar_6(float(fim - inicio))
    }

    resumo_combinacoes.append(nova_linha)

    if verbose:
        print(f"  - Finalizada em {fim - inicio:.2f} segundos")
        print(f"  - Score_Final: {score_final:.6f}")
        print(f"  - Neg_Log_Veross_Com_Rotulo: {neg_log_veross_com_rotulo:.6f}")
        print(f"  - Neg_Log_Veross_GMM: {neg_log_veross_gmm:.6f}")
        print(f"  - Diferenca_Neg_Log_Veross: {diferenca_neg_log_veross:.6f}")

    return resumo_combinacoes

resumo_combinacoes = []

inicio_geral = time.perf_counter()

for feature_1, feature_2 in tqdm(
    combinacoes_2x2,
    desc="Processando combinações 2x2",
    unit="combinação"
):
    tamanho_antes = len(resumo_combinacoes)

    resumo_combinacoes = analisar_combinacao_2x2(
        df=df,
        feature_1=feature_1,
        feature_2=feature_2,
        target_name=target_name,
        resumo_combinacoes=resumo_combinacoes,
        verbose=False
    )

    tamanho_depois = len(resumo_combinacoes)

    if tamanho_depois > tamanho_antes:
        tqdm.write(f"Combinação processada: {feature_1} + {feature_2}")
    else:
        tqdm.write(f"Combinação ignorada ou com erro: {feature_1} + {feature_2}")

fim_geral = time.perf_counter()

scores_2x2 = pd.DataFrame(resumo_combinacoes)

if scores_2x2.empty:
    raise ValueError(
        "Nenhuma combinação 2x2 foi processada. "
        "Verifique se existem features numéricas válidas e se o target está correto."
    )

scores_2x2 = scores_2x2.sort_values(
    by="Score_Final",
    ascending=False
).reset_index(drop=True)

scores_2x2["Posicao_Rank"] = np.arange(
    1,
    len(scores_2x2) + 1
)

scores_2x2 = scores_2x2[
    [
        "Feature_1",
        "Feature_2",
        "Combinacao",
        "AUC_PR",
        "MCC",
        "Log_Loss",
        "Log_Loss_Norm",
        "Neg_Log_Veross_Com_Rotulo",
        "Neg_Log_Veross_GMM",
        "Diferenca_Neg_Log_Veross",
        "Score_Final",
        "Melhor_Ponto_Corte",
        "Ponto_Corte_Medio",
        "Tempo",
        "Posicao_Rank"
    ]
]


scores_2x2.to_csv(
    "2x2_scores.csv",
    index=False,
    encoding="utf-8-sig",
    float_format="%.6f"
)

tempo_total_segundos = fim_geral - inicio_geral
tempo_total_minutos = tempo_total_segundos / 60

print("\nProcessamento finalizado.")
print(f"Combinações processadas com sucesso: {len(scores_2x2)}")
print(f"Tempo total: {tempo_total_segundos:.2f} segundos")
print(f"Tempo total: {tempo_total_minutos:.2f} minutos")
print("Arquivo salvo como: 2x2_scores.csv")

print("\nTop 20 combinações:")
display(scores_2x2.head(20))